# Slice browser + plotting + native-resolution PNG export (clean)

This notebook does four things:

1. **Discovers cases** from `adrienParamClass.py` (or, if that fails, from folders named like `R1P7` under a project root).
2. For each case, lists available **3D Fortran binaries** (`u,v,w,r,ee,chi`) and their sizes.
3. Scans `001_Final/2D_slices` (or `2D_Slices`) for **NetCDF slices** and builds a catalog:
   - slices saved *all-in-one* NetCDF (contains multiple variables), or
   - slices saved *var-by-var* (one variable per NetCDF).
4. Lets you pick any **complete slice set** (all of `u,v,w,r,ee,chi` available for the same plane+index+stride), then:
   - plots the usual 6-panel diagnostic figure,
   - exports native-resolution PNGs into `./figures/<case>/<plane>_<axis><idx>/`.

> Everything exports relative to the directory where this notebook is running.


In [2]:
# Core imports
from __future__ import annotations

import os
import re
import math
import json
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.io import netcdf_file
from PIL import Image

import ipywidgets as widgets
from IPython.display import display, clear_output


In [3]:
# Paths / configuration

NOTEBOOK_ROOT = Path.cwd()
FIGURES_ROOT = NOTEBOOK_ROOT / "figures"
FIGURES_ROOT.mkdir(exist_ok=True)

# If you know the project root, set it here.
# Otherwise we will try some common paths, then fall back to searching upward from cwd.
PROJECT_ROOT_CANDIDATES = [
    Path("/lustre/orion/cfd135/proj-shared/Hsst"),
    Path("/lustre/orion/cfd135/proj-shared/Hsstchgrp"),
    Path("/lustre/orion/cfd135/proj-shared"),
    NOTEBOOK_ROOT,
]

# If you know exactly where adrienParamClass.py is, set it here.
# Otherwise we will try to auto-find it.
PARAM_PATH_HINTS = [
    NOTEBOOK_ROOT / "adrienParamClass.py",
    NOTEBOOK_ROOT / "adrienParamClass" / "adrienParamClass.py",
]

REQ_VARS = ["u", "v", "w", "r", "ee", "chi"]
CASE_RE = re.compile(r"^R\d+P\d+$")

SLICE_DIR_CANDIDATES = ["2D_slices", "2D_Slices", "2D_SLICES"]

print("Notebook root:", NOTEBOOK_ROOT)
print("Figures root :", FIGURES_ROOT)


Notebook root: /autofs/nccs-svm1_home2/lefauve/git/INCITE/adrien
Figures root : /autofs/nccs-svm1_home2/lefauve/git/INCITE/adrien/figures


In [4]:
# Utilities: formatting + sizes

def human_bytes(n: int) -> str:
    if n is None:
        return ""
    n = float(n)
    for unit in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024.0:
            return f"{n:,.1f} {unit}"
        n /= 1024.0
    return f"{n:,.1f} EB"

def safe_stat(path: Path):
    try:
        st = path.stat()
        return st
    except FileNotFoundError:
        return None


In [10]:
# Step 1: locate adrienParamClass.py and the project root

def find_first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        if p and p.exists():
            return p
    return None

def find_param_file() -> Optional[Path]:
    # 1) direct hints
    p = find_first_existing(PARAM_PATH_HINTS)
    if p:
        return p

    # 2) search upward from cwd
    here = NOTEBOOK_ROOT
    for parent in [here] + list(here.parents):
        cand = parent / "adrienParamClassSheared.py"
        if cand.exists():
            return cand

    # 3) try deeper search (limited)
    # Walk only a couple levels to avoid scanning huge lustre trees.
    max_depth = 3
    for base in [NOTEBOOK_ROOT] + list(NOTEBOOK_ROOT.parents[:3]):
        try:
            for root, dirs, files in os.walk(base):
                rootp = Path(root)
                depth = len(rootp.relative_to(base).parts)
                if depth > max_depth:
                    dirs[:] = []
                    continue
                if "adrienParamClassSheared.py" in files:
                    return rootp / "adrienParamClassSheared.py"
        except Exception:
            continue
    return None

def find_project_root() -> Optional[Path]:
    for cand in PROJECT_ROOT_CANDIDATES:
        if cand.exists():
            # If it contains case-like folders, good.
            try:
                kids = [p for p in cand.iterdir() if p.is_dir() and CASE_RE.match(p.name)]
                if kids:
                    return cand
            except Exception:
                pass
    # fallback: climb upward from cwd looking for case-like dirs
    for parent in [NOTEBOOK_ROOT] + list(NOTEBOOK_ROOT.parents):
        try:
            kids = [p for p in parent.iterdir() if p.is_dir() and CASE_RE.match(p.name)]
            if kids:
                return parent
        except Exception:
            continue
    return None

PARAM_PATH = find_param_file()
PROJECT_ROOT = find_project_root()

print("adrienParamClassSheared.py:", PARAM_PATH if PARAM_PATH else "NOT FOUND (will fall back to folder scan)")
print("Project root       :", PROJECT_ROOT if PROJECT_ROOT else "NOT FOUND (set PROJECT_ROOT_CANDIDATES manually)")


adrienParamClassSheared.py: /autofs/nccs-svm1_home2/lefauve/git/INCITE/adrien/adrienParamClassSheared.py
Project root       : /lustre/orion/cfd135/proj-shared/Hsst


In [11]:
# Step 2: import adrienParamClassSheared.py (if present) and build a case -> param mapping (best-effort)

import importlib.util

def import_module_from_path(pyfile: Path, module_name="adrienParamClass_local"):
    spec = importlib.util.spec_from_file_location(module_name, str(pyfile))
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod

MOD = None
if PARAM_PATH and PARAM_PATH.exists():
    try:
        MOD = import_module_from_path(PARAM_PATH)
        print("Imported:", PARAM_PATH)
    except Exception as e:
        MOD = None
        print("Failed to import adrienParamClass.py:", e)

# Step 2: import adrienParamClass*.py and build a case -> DatParam mapping (robust)

import importlib.util
from pathlib import Path
from typing import Dict

def import_module_from_path(pyfile: Path, module_name="adrienParamClass_local"):
    spec = importlib.util.spec_from_file_location(module_name, str(pyfile))
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod

MOD = None
if PARAM_PATH and PARAM_PATH.exists():
    try:
        MOD = import_module_from_path(PARAM_PATH)
        print("Imported:", PARAM_PATH)
    except Exception as e:
        MOD = None
        print("Failed to import param file:", e)

def build_params_by_case(mod) -> Dict[str, object]:
    """
    Your param file defines generate() that returns {'R1P1': DatParam, ...}.
    Use that as the source of truth.
    Fallbacks exist in case the file changes later.
    """
    if mod is None:
        return {}

    # 1) Preferred: generate()
    if hasattr(mod, "generate"):
        try:
            d = mod.generate()
            if isinstance(d, dict) and d:
                out = {str(k): v for k, v in d.items()}
                # sanity: keep only param-like objects
                out = {k: v for k, v in out.items() if hasattr(v, "name") and hasattr(v, "dirPath")}
                return out
        except Exception as e:
            print("Warning: calling generate() failed:", e)

    # 2) Fallback: search module globals for param-like objects
    params: Dict[str, object] = {}
    for _, obj in vars(mod).items():
        if hasattr(obj, "name") and hasattr(obj, "dirPath"):
            cname = str(getattr(obj, "name"))
            if CASE_RE.match(cname):
                params[cname] = obj

    # 3) Fallback: dicts/lists that contain param-like objects
    for _, obj in vars(mod).items():
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(k, str) and hasattr(v, "name") and hasattr(v, "dirPath"):
                    if CASE_RE.match(k) or CASE_RE.match(str(v.name)):
                        params[str(v.name)] = v
        if isinstance(obj, (list, tuple)):
            for v in obj:
                if hasattr(v, "name") and hasattr(v, "dirPath"):
                    cname = str(v.name)
                    if CASE_RE.match(cname):
                        params[cname] = v

    return params

PARAMS_BY_CASE = build_params_by_case(MOD)
print("Params found for cases:", len(PARAMS_BY_CASE))
print("Cases:", sorted(PARAMS_BY_CASE.keys())[:30], "..." if len(PARAMS_BY_CASE) > 30 else "")

# Optional: infer project root once, if you want it
if len(PARAMS_BY_CASE) and PROJECT_ROOT is None:
    try:
        anyp = next(iter(PARAMS_BY_CASE.values()))
        # your dirPath ends with .../R?P?/001_Final/
        PROJECT_ROOT = Path(anyp.dirPath).resolve().parents[1]
        print("Inferred project root:", PROJECT_ROOT)
    except Exception:
        pass

Imported: /autofs/nccs-svm1_home2/lefauve/git/INCITE/adrien/adrienParamClassSheared.py
Imported: /autofs/nccs-svm1_home2/lefauve/git/INCITE/adrien/adrienParamClassSheared.py
Params found for cases: 12
Cases: ['R10P1', 'R10P7', 'R1P1', 'R1P50', 'R1P7', 'R4P1', 'R4P50', 'R4P7', 'R6P1', 'R6P7', 'R8P1', 'R8P7'] 


In [12]:
# Step 3: discover case folders

def discover_cases(project_root: Optional[Path]) -> List[str]:
    cases = set(PARAMS_BY_CASE.keys())
    if project_root and project_root.exists():
        try:
            for p in project_root.iterdir():
                if p.is_dir() and CASE_RE.match(p.name):
                    cases.add(p.name)
        except Exception as e:
            print("Could not scan project root:", e)
    return sorted(cases)

CASES = discover_cases(PROJECT_ROOT)
print("Cases discovered:", len(CASES))
print(CASES[:20], "..." if len(CASES)>20 else "")


Cases discovered: 15
['R10P1', 'R10P50', 'R10P7', 'R1P1', 'R1P50', 'R1P7', 'R4P1', 'R4P50', 'R4P7', 'R6P1', 'R6P50', 'R6P7', 'R8P1', 'R8P50', 'R8P7'] 


In [32]:
# Step 4: scan 3D Fortran binaries for each case and present as a table

def case_base_dir(case: str) -> Path:
    # Prefer params if they exist (dirPath often points to the data folder directly)
    if case in PARAMS_BY_CASE:
        try:
            return Path(PARAMS_BY_CASE[case].dirPath)
        except Exception:
            pass
    # fallback: PROJECT_ROOT/<case>
    if PROJECT_ROOT:
        return PROJECT_ROOT / case
    raise RuntimeError("No PROJECT_ROOT and no PARAMS for this case.")

def scan_binaries_for_case(case: str) -> pd.DataFrame:
    base = case_base_dir(case)
    rows = []
    if not base.exists():
        return pd.DataFrame(columns=["case","dir","var","file","bytes","size"])

    for var in REQ_VARS:
        search_dirs = [base, base/"001_Final", base/"001_Final"/"3D", base/"3D", base/"output"]
        seen = set()
        for d in search_dirs:
            if not d.exists():
                continue
            for f in d.glob(f"{var}_*"):
                if not f.is_file():
                    continue
                if f.name.endswith(".cksums") or f.name.endswith(".cksum"):   # ← exclude checksum files
                    continue
                if f in seen:
                    continue

                seen.add(f)
                st = safe_stat(f)
                rows.append({
                    "case": case,
                    "dir": str(f.parent),
                    "var": var,
                    "file": f.name,
                    "bytes": st.st_size if st else None,
                    "size": human_bytes(st.st_size) if st else "",
                })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["case","var","bytes"], ascending=[True, True, False])
    return df

# Build once for all cases (can be large; feel free to limit if needed)
all_bin = []
for c in CASES:
    all_bin.append(scan_binaries_for_case(c))
BIN_DF = pd.concat(all_bin, ignore_index=True) if all_bin else pd.DataFrame()

# BIN_DF.head()
# BIN_DF = BIN_DF.sort_values(["case","var","file"]).reset_index(drop=True)
# BIN_DF.head(100)

from IPython.display import display

display(
    BIN_DF.style
        .set_table_attributes('style="display:block; max-height:600px; overflow:auto;"')
)

,case,dir,var,file,bytes,size
0,R10P1,/lustre/orion/cfd135/proj-shared/Hsst/R10P1/001_Final,r,r_198.101596,927863930880,864.1 GB
1,R10P1,/lustre/orion/cfd135/proj-shared/Hsst/R10P1/001_Final,u,u_198.101596,927863930880,864.1 GB
2,R10P1,/lustre/orion/cfd135/proj-shared/Hsst/R10P1/001_Final,v,v_198.101596,927863930880,864.1 GB
3,R10P1,/lustre/orion/cfd135/proj-shared/Hsst/R10P1/001_Final,w,w_198.101596,927863930880,864.1 GB
4,R10P7,/lustre/orion/cfd135/proj-shared/Hsst/R10P7/001_Final,r,r_48.210113,15898382438400,14.5 TB
5,R10P7,/lustre/orion/cfd135/proj-shared/Hsst/R10P7/001_Final,u,u_48.210113,15898382438400,14.5 TB
6,R10P7,/lustre/orion/cfd135/proj-shared/Hsst/R10P7/001_Final,v,v_48.210113,15898382438400,14.5 TB
7,R10P7,/lustre/orion/cfd135/proj-shared/Hsst/R10P7/001_Final,w,w_48.210113,15898382438400,14.5 TB
8,R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final,chi,chi_1100.000000,1814298624,1.7 GB
9,R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final,ee,ee_1100.000000,1814298624,1.7 GB


In [33]:
BIN_DF.groupby(["case","var"]).size().unstack(fill_value=0)

var,chi,ee,r,u,v,w
case,,,,,,
R10P1,0,0,1,1,1,1
R10P7,0,0,1,1,1,1
R1P1,1,1,1,1,1,1
R1P50,1,1,1,1,1,1
R1P7,1,1,1,1,1,1
R4P1,1,1,1,1,1,1
R4P50,1,1,1,1,1,1
R4P7,1,1,1,1,1,1
R6P1,1,1,1,1,1,1


In [14]:
# Nicely present: binaries per case, with total size per var

if BIN_DF.empty:
    display(pd.DataFrame({"note":["No binaries found (or paths not accessible)."]}))
else:
    summary = (BIN_DF.groupby(["case","var"], as_index=False)
               .agg(nfiles=("file","count"), bytes=("bytes","sum")))
    summary["size"] = summary["bytes"].apply(human_bytes)
    display(summary.sort_values(["case","var"]))


,case,var,nfiles,bytes,size
0,R10P1,r,1,927863930880,864.1 GB
1,R10P1,u,1,927863930880,864.1 GB
2,R10P1,v,1,927863930880,864.1 GB
3,R10P1,w,1,927863930880,864.1 GB
4,R10P7,r,1,15898382438400,14.5 TB
...,...,...,...,...,...
65,R8P1,w,1,333432013824,310.5 GB
66,R8P7,r,2,12231652147200,11.1 TB
67,R8P7,u,2,12231652147200,11.1 TB
68,R8P7,v,2,12231652147200,11.1 TB


In [39]:
# Step 5: scan 2D slice NetCDFs per case and build a catalog

SLICE_NAME_RE = re.compile(
    r"^(?P<case>R\d+P\d+)_(?P<plane>xy|xz|yz)_(?P<axis>[xyz])(?P<idx>\d+)_st(?P<stride>\d+x\d+)(?:_(?P<var>u|v|w|r|ee|chi))?\.nc$"
)

def find_slice_dir(case: str) -> Optional[Path]:
    base = case_base_dir(case)

    # base might already be .../001_Final
    if base.name == "001_Final":
        final = base
    else:
        final = base / "001_Final"

    for dn in SLICE_DIR_CANDIDATES:
        p = final / dn
        if p.exists():
            return p
    return None

def nc_vars(path: Path) -> List[str]:
    try:
        nc = netcdf_file(str(path), "r", mmap=False)
        try:
            return list(nc.variables.keys())
        finally:
            nc.close()
    except Exception:
        return []

def scan_slices_for_case(case: str) -> pd.DataFrame:
    sdir = find_slice_dir(case)
    rows = []
    if sdir is None:
        return pd.DataFrame(columns=[
            "case","slice_dir","file","plane","axis","idx","stride","mode",
            "var","vars_in_file","bytes","size"
        ])

    for f in sorted(sdir.glob("*.nc")):
        m = SLICE_NAME_RE.match(f.name)
        st = safe_stat(f)
        vars_in = nc_vars(f)
        vars_in_clean = [v for v in vars_in if v not in {"x","y","z","x0","y0","z0","ix","iy","iz"}]

        if m:
            gd = m.groupdict()
            plane = gd["plane"]
            axis = gd["axis"]
            idx = int(gd["idx"])
            stride = gd["stride"]
            var = gd.get("var")
            mode = "split" if var else "combined"
        else:
            # nonconforming filename: treat as unknown
            plane = axis = None
            idx = None
            stride = None
            var = None
            mode = "unknown"

        rows.append({
            "case": case,
            "slice_dir": str(sdir),
            "file": f.name,
            "plane": plane,
            "axis": axis,
            "idx": idx,
            "stride": stride,
            "mode": mode,
            "var": var,
            "vars_in_file": ",".join(vars_in_clean),
            "bytes": st.st_size if st else None,
            "size": human_bytes(st.st_size) if st else "",
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["case","plane","idx","var","file"], na_position="last")
    return df

all_slice = []
for c in CASES:
    dfc = scan_slices_for_case(c)
    if dfc is not None and not dfc.empty:
        all_slice.append(dfc)

SLICE_DF = pd.concat(all_slice, ignore_index=True) if all_slice else pd.DataFrame()

from IPython.display import display, HTML

def show_scroll(df, max_height_px=500):
    display(HTML(f"""
    <div style="max-height:{max_height_px}px; overflow:auto; border:1px solid #ddd;">
      {df.to_html(index=False)}
    </div>
    """))

show_scroll(SLICE_DF)

/tmp/ipykernel_1134/3248752134.py:89: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  SLICE_DF = pd.concat(all_slice, ignore_index=True) if all_slice else pd.DataFrame()


case,slice_dir,file,plane,axis,idx,stride,mode,var,vars_in_file,bytes,size
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z55_st1x1.nc,xy,z,55.0,1x1,combined,None,"u,v,w,r,ee,chi",28330812,27.0 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z77_st1x1.nc,xy,z,77.0,1x1,combined,None,"u,v,w,r,ee,chi",28330812,27.0 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z110_st1x1.nc,xy,z,110.0,1x1,combined,None,"u,v,w,r,ee,chi",28330812,27.0 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z154_st1x1.nc,xy,z,154.0,1x1,combined,None,"u,v,w,r,ee,chi",28330812,27.0 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z165_st1x1.nc,xy,z,165.0,1x1,combined,None,"u,v,w,r,ee,chi",28330812,27.0 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z192_st1x1_chi.nc,xy,z,192.0,1x1,split,chi,chi,4737652,4.5 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z192_st1x1_ee.nc,xy,z,192.0,1x1,split,ee,ee,4737652,4.5 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z192_st1x1_r.nc,xy,z,192.0,1x1,split,r,r,4737652,4.5 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z192_st1x1_u.nc,xy,z,192.0,1x1,split,u,u,4737652,4.5 MB
R1P1,/lustre/orion/cfd135/proj-shared/Hsst/R1P1/001_Final/2D_slices,R1P1_xy_z192_st1x1_v.nc,xy,z,192.0,1x1,split,v,v,4737652,4.5 MB


In [40]:
# Present slices nicely (grouped)

if SLICE_DF.empty:
    display(pd.DataFrame({"note":["No slice NetCDFs found."]}))
else:
    # quick view: counts by case/plane/mode
    counts = (SLICE_DF.dropna(subset=["plane","idx"])
              .groupby(["case","plane","mode"], as_index=False)
              .agg(nfiles=("file","count"), total_bytes=("bytes","sum")))
    counts["total_size"] = counts["total_bytes"].apply(human_bytes)
    display(counts.sort_values(["case","plane","mode"]))


,case,plane,mode,nfiles,total_bytes,total_size
0,R1P1,xy,combined,11,311638932,297.2 MB
1,R1P1,xy,split,6,28425912,27.1 MB
2,R1P1,xz,combined,17,240923388,229.8 MB
3,R1P1,yz,combined,1,0,0.0 B
4,R1P7,xy,combined,16,1812542400,1.7 GB
5,R1P7,xy,split,30,567355800,541.1 MB
6,R1P7,xz,combined,16,906474432,864.5 MB
7,R1P7,xz,split,6,56811192,54.2 MB
8,R6P7,xy,split,30,9064139160,8.4 GB
9,R6P7,yz,combined,1,0,0.0 B


In [41]:
# Step 6: build "complete slice sets" index (combined or split) for dropdown selection

@dataclass(frozen=True)
class SliceKey:
    case: str
    plane: str
    axis: str
    idx: int
    stride: str

def build_complete_sets(slice_df: pd.DataFrame) -> pd.DataFrame:
    if slice_df.empty:
        return pd.DataFrame()

    df = slice_df.dropna(subset=["case","plane","axis","idx","stride"]).copy()

    # For combined files: vars available are those inside the file
    combined = df[df["mode"]=="combined"].copy()
    combined["available"] = combined["vars_in_file"].apply(lambda s: sorted([v for v in s.split(",") if v]))

    # For split files: group files by key and collect vars by filename suffix
    split = df[df["mode"]=="split"].copy()

    # Combined: one row already corresponds to a slice key
    comb_rows = []
    for _, r in combined.iterrows():
        key = SliceKey(r["case"], r["plane"], r["axis"], int(r["idx"]), r["stride"])
        avail = set(r["available"])
        # Some combined files might store 'ee' and 'chi' under different names; we only accept exact names here.
        ok = all(v in avail for v in REQ_VARS)
        comb_rows.append({
            "case": key.case, "plane": key.plane, "axis": key.axis, "idx": key.idx, "stride": key.stride,
            "storage": "combined",
            "ok": ok,
            "files": json.dumps({"combined": str(Path(r["slice_dir"])/r["file"])}),
            "available_vars": ",".join(sorted(avail)),
        })
    comb_df = pd.DataFrame(comb_rows)

    # Split: group
    split_rows = []
    if not split.empty:
        for (case, plane, axis, idx, stride), g in split.groupby(["case","plane","axis","idx","stride"]):
            avail = set([vv for vv in g["var"].tolist() if isinstance(vv, str)])
            ok = all(v in avail for v in REQ_VARS)
            filemap = {v: str(Path(g.iloc[0]["slice_dir"])/g[g["var"]==v].iloc[0]["file"]) for v in avail}
            split_rows.append({
                "case": case, "plane": plane, "axis": axis, "idx": int(idx), "stride": stride,
                "storage": "split",
                "ok": ok,
                "files": json.dumps(filemap),
                "available_vars": ",".join(sorted(avail)),
            })
    split_df = pd.DataFrame(split_rows)

    out = pd.concat([comb_df, split_df], ignore_index=True) if (not comb_df.empty or not split_df.empty) else pd.DataFrame()
    if not out.empty:
        out = out.sort_values(["ok","case","plane","idx","stride","storage"], ascending=[False, True, True, True, True, True])
    return out

COMPLETE_DF = build_complete_sets(SLICE_DF)

if COMPLETE_DF.empty:
    display(pd.DataFrame({"note":["No complete slice sets detected yet."]}))
else:
    display(COMPLETE_DF.head(30))
    print("Complete sets:", int(COMPLETE_DF["ok"].sum()), "/", len(COMPLETE_DF))


,case,plane,axis,idx,stride,storage,ok,files,available_vars
0,R1P1,xy,z,55,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
1,R1P1,xy,z,77,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
2,R1P1,xy,z,110,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
3,R1P1,xy,z,154,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
4,R1P1,xy,z,165,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
5,R1P1,xy,z,192,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
63,R1P1,xy,z,192,1x1,split,True,"{""v"": ""/lustre/orion/cfd135/proj-shared/Hsst/R...","chi,ee,r,u,v,w"
6,R1P1,xy,z,219,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
7,R1P1,xy,z,230,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"
8,R1P1,xy,z,274,1x1,combined,True,"{""combined"": ""/lustre/orion/cfd135/proj-shared...","chi,ee,r,u,v,w"


Complete sets: 72 / 75


In [44]:
# Plotting + export helpers (2D raw slices)

def imshow_with_cbar(ax, Z, cmap, vmin, vmax, cbar_label):
    im = ax.imshow(
        Z.T,
        origin="lower",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="equal",
    )
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3.5%", pad=0.08)
    cb = ax.figure.colorbar(im, cax=cax)
    cb.set_ticks([vmin, 0, vmax] if (vmin < 0 < vmax) else [vmin, vmax])
    cb.set_label(cbar_label)
    return im, cb

def set_index_axis(ax, axis="x", N=1, label=None, step=500, label_step=1000):
    N = int(N)
    ticks = list(range(0, N, step))
    labels = [str(t) if (t % label_step == 0) else "" for t in ticks]
    if axis == "x":
        ax.set_xticks(ticks); ax.set_xticklabels(labels)
        if label is not None: ax.set_xlabel(label)
    elif axis == "y":
        ax.set_yticks(ticks); ax.set_yticklabels(labels)
        if label is not None: ax.set_ylabel(label)
    else:
        raise ValueError("axis must be 'x' or 'y'")

def save_field_native_pixels(
    Z,
    path,
    cmap="viridis",
    vmin=None,
    vmax=None,
    nan_color=(0, 0, 0, 0),
):
    Z = np.asarray(Z)
    if Z.ndim != 2:
        raise ValueError(f"Z must be 2D, got shape {Z.shape}")
    if vmin is None:
        vmin = np.nanmin(Z)
    if vmax is None:
        vmax = np.nanmax(Z)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        raise ValueError(f"Bad vmin/vmax: vmin={vmin}, vmax={vmax}")

    cmap_obj = cm.get_cmap(cmap) if isinstance(cmap, str) else cmap
    norm = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)

    rgba = cmap_obj(norm(Z))  # float in [0,1]
    nan_mask = ~np.isfinite(Z)
    if np.any(nan_mask):
        r, g, b, a = [c / 255.0 for c in nan_color]
        rgba[nan_mask] = (r, g, b, a)

    rgba8 = (rgba * 255).astype(np.uint8)
    Image.fromarray(rgba8, mode="RGBA").save(str(path))
    return Path(path)

def plane_stats(raw2d: Dict[str, np.ndarray], g: float, N2: float) -> dict:
    u = raw2d["u"].astype(np.float64, copy=False)
    v = raw2d["v"].astype(np.float64, copy=False)
    w = raw2d["w"].astype(np.float64, copy=False)
    r = raw2d["r"].astype(np.float64, copy=False)
    ee  = raw2d["ee"].astype(np.float64, copy=False)
    chi = raw2d["chi"].astype(np.float64, copy=False)

    Ek = 0.5 * np.mean(u*u + v*v + w*w)
    Ep = np.mean(r*r) * (g**2) / (2.0 * N2) if N2 != 0 else np.nan
    eps_avg = float(np.mean(ee))
    chi_avg = float(np.mean(chi))
    return dict(Ek=float(Ek), Ep=float(Ep), eps_avg=eps_avg, chi_avg=chi_avg)

def plot_6panel_from_raw2d(case: str, plane: str, axis: str, idx: int, raw2d: Dict[str,np.ndarray], g=9.81, N2=1.0):
    stats = plane_stats(raw2d, g=g, N2=N2)
    Ek, Ep = stats["Ek"], stats["Ep"]
    eps_avg, chi_avg = stats["eps_avg"], stats["chi_avg"]

    with np.errstate(divide="ignore", invalid="ignore"):
        U = raw2d["u"] / np.sqrt(Ek)
        V = raw2d["v"] / np.sqrt(Ek)
        W = raw2d["w"] / np.sqrt(Ek)
        B = (raw2d["r"] * (-g)) / np.sqrt(N2 * Ep) if (N2>0 and Ep>0) else np.full_like(raw2d["r"], np.nan, dtype=np.float32)
        E = np.log10(raw2d["ee"] / eps_avg)
        C = np.log10(raw2d["chi"] / chi_avg)

    fig, axs = plt.subplots(6, 1, figsize=(10, 16))
    fig.subplots_adjust(hspace=0.55)

    panels = [
        (U, "seismic", -5, 5, r"$u'/E_k^{1/2}$", "u"),
        (V, "seismic", -5, 5, r"$v'/E_k^{1/2}$", "v"),
        (W, "seismic", -5, 5, r"$w'/E_k^{1/2}$", "w"),
        (B, "viridis", -5, 5, r"$b'/(N E_p^{1/2})$", "b"),
        (E, "magma",   -2, 2, r"$\log_{10}(\varepsilon/\langle\varepsilon\rangle)$", "eps"),
        (C, "hot",     -2, 3, r"$\log_{10}(\chi/\langle\chi\rangle)$", "chi"),
    ]

    title_plane = {"xy":"(x,y)","xz":"(x,z)","yz":"(y,z)"}[plane]
    fixed = f"{axis}{idx}"

    for i, (Z, cmap, vmin, vmax, cbar_lab, short) in enumerate(panels):
        imshow_with_cbar(axs[i], Z, cmap, vmin, vmax, cbar_lab)
        axs[i].set_title(f"{case} {short} {title_plane} at {fixed}")

        Nx_plot, Ny_plot = Z.shape
        # labels depend on plane
        if plane == "xy":
            xlab, ylab = "x index", "y index"
        elif plane == "xz":
            xlab, ylab = "x index", "z index"
        else:
            xlab, ylab = "y index", "z index"

        set_index_axis(axs[i], "x", Nx_plot, " " if i < 4 else xlab)
        set_index_axis(axs[i], "y", Ny_plot, ylab)

    return fig, axs, stats

def export_native_pngs(case: str, plane: str, axis: str, idx: int, raw2d: Dict[str,np.ndarray], g=9.81, N2=1.0):
    stats = plane_stats(raw2d, g=g, N2=N2)
    Ek, Ep = stats["Ek"], stats["Ep"]
    eps_avg, chi_avg = stats["eps_avg"], stats["chi_avg"]

    with np.errstate(divide="ignore", invalid="ignore"):
        U = raw2d["u"] / np.sqrt(Ek)
        V = raw2d["v"] / np.sqrt(Ek)
        W = raw2d["w"] / np.sqrt(Ek)
        B = (raw2d["r"] * (-g)) / np.sqrt(N2 * Ep) if (N2>0 and Ep>0) else np.full_like(raw2d["r"], np.nan, dtype=np.float32)
        E = np.log10(raw2d["ee"] / eps_avg)
        C = np.log10(raw2d["chi"] / chi_avg)

    # Export dir: figures/<case>/<plane>_<axis><idx>/
    outdir = FIGURES_ROOT / case / f"{plane}_{axis}{idx}"
    outdir.mkdir(parents=True, exist_ok=True)

    def X(Z):  # match imshow(Z.T, origin=lower)
        return np.flipud(Z.T)

    items = [
        ("u",   X(U), "seismic", -5, 5),
        ("v",   X(V), "seismic", -5, 5),
        ("w",   X(W), "seismic", -5, 5),
        ("b",   X(B), "viridis", -5, 5),
        ("eps", X(E), "magma",   -2, 2),
        ("chi", X(C), "hot",     -2, 3),
    ]

    paths = {}
    for name, Z2, cmap, vmin, vmax in items:
        path = outdir / f"{name}.png"
        save_field_native_pixels(Z2, path, cmap=cmap, vmin=vmin, vmax=vmax)
        paths[name] = str(path)

    return outdir, paths, stats


In [45]:
# NetCDF readers for combined or split files

def read_vars_from_nc(path: Path, varnames: List[str]) -> Dict[str, np.ndarray]:
    out = {}
    nc = netcdf_file(str(path), "r", mmap=False)
    try:
        for v in varnames:
            if v not in nc.variables:
                raise KeyError(f"{path.name}: missing variable '{v}'. Available: {list(nc.variables.keys())}")
            out[v] = np.array(nc.variables[v][:], dtype=np.float32)
    finally:
        nc.close()
    return out

def load_slice_set(row: pd.Series) -> Tuple[Dict[str,np.ndarray], Dict]:
    """Return raw2d dict and a small meta dict."""
    files = json.loads(row["files"])
    meta = {"storage": row["storage"], "files": files}

    if row["storage"] == "combined":
        p = Path(files["combined"])
        raw2d = read_vars_from_nc(p, REQ_VARS)
        return raw2d, meta

    if row["storage"] == "split":
        raw2d = {}
        for v in REQ_VARS:
            p = Path(files[v])
            raw2d[v] = read_vars_from_nc(p, [v])[v]
        return raw2d, meta

    raise ValueError(f"Unknown storage: {row['storage']}")


In [48]:
# Try to fetch g and N2 from params (best-effort), otherwise fall back.

def get_g_N2(case: str) -> Tuple[float, float]:
    g = 9.81
    N2 = 1.0
    if case in PARAMS_BY_CASE:
        p = PARAMS_BY_CASE[case]
        if hasattr(p, "zAccel"):
            g = float(p.zAccel)
        if hasattr(p, "N2"):
            N2 = float(p.N2)
        elif hasattr(p, "dGrad") and hasattr(p, "zAccel"):
            # sometimes N2 = -dGrad * zAccel
            try:
                N2 = float(-p.dGrad * p.zAccel)
            except Exception:
                pass
    return g, N2

for c in CASES:
    g, N2 = get_g_N2(c)
    print(f"{c:6s}  g={g:.3g}  N2={N2:.3g}")


R10P1   g=160  N2=0.16
R10P50  g=9.81  N2=1
R10P7   g=160  N2=0.16
R1P1    g=164  N2=0.164
R1P50   g=167  N2=0.167
R1P7    g=153  N2=0.153
R4P1    g=160  N2=0.16
R4P50   g=167  N2=0.167
R4P7    g=153  N2=0.153
R6P1    g=152  N2=0.152
R6P50   g=9.81  N2=1
R6P7    g=152  N2=0.152
R8P1    g=150  N2=0.15
R8P50   g=9.81  N2=1
R8P7    g=160  N2=0.16


In [49]:
# Step 7: Interactive browser (dropdown + buttons)

if COMPLETE_DF.empty:
    print("No slice sets found. Check PROJECT_ROOT/PARAM_PATH and re-run the scan cells above.")
else:
    ok_df = COMPLETE_DF[COMPLETE_DF["ok"]].copy()

    def label_row(r):
        return f"{r['case']} | {r['plane']} {r['axis']}{int(r['idx'])} | stride {r['stride']} | {r['storage']}"

    options = [(label_row(r), i) for i, r in ok_df.reset_index(drop=True).iterrows()]

    dd = widgets.Dropdown(
        options=options,
        description="Slice:",
        layout=widgets.Layout(width="98%")
    )

    btn_plot = widgets.Button(description="Plot 6-panel", button_style="primary")
    btn_export = widgets.Button(description="Export native PNGs", button_style="success")
    out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))

    def _get_selected():
        idx = dd.value
        row = ok_df.reset_index(drop=True).iloc[idx]
        return row

    def on_plot(_):
        with out:
            clear_output(wait=True)
            row = _get_selected()
            raw2d, meta = load_slice_set(row)
            g, N2 = get_g_N2(row['case'])
            fig, axs, stats = plot_6panel_from_raw2d(
                row["case"], row["plane"], row["axis"], int(row["idx"]),
                raw2d, g=g, N2=N2
            )
            print("Meta:", meta)
            print("Stats (from this 2D slice):", stats)
            plt.show()

    def on_export(_):
        with out:
            clear_output(wait=True)
            row = _get_selected()
            raw2d, meta = load_slice_set(row)
            g, N2 = get_g_N2(row['case'])
            outdir, paths, stats = export_native_pngs(
                row["case"], row["plane"], row["axis"], int(row["idx"]),
                raw2d, g=g, N2=N2
            )
            print("Meta:", meta)
            print("Stats (from this 2D slice):", stats)
            print("Exported to:", outdir)
            display(pd.DataFrame({"var": list(paths.keys()), "png": list(paths.values())}))

    btn_plot.on_click(on_plot)
    btn_export.on_click(on_export)

    display(widgets.VBox([dd, widgets.HBox([btn_plot, btn_export]), out]))
